# Jax Gradient Manipulations

## Lesson Goals:

By the end of this lesson you will understand gradient manipulations: how to stop gradient calculations from flowing backwards through parts of our graph, and how to stop skip applying gradients.

## Core Concepts:

- gradient stopping

- skipping gradient application

## Related Notebooks

- [Gradient Basics](./exe_07_grad_basics.ipynb) - basic derivatives
- [Advanced Autodiff](./exe_10_grad_advanced.ipynb) - custom derivatives

## Additional Reading:

- [Jax Documentation: Custom Derivative Rules](https://docs.jax.dev/en/latest/notebooks/Custom_derivative_rules_for_Python_code.html#summary)
- [Jax Documentation: Advanced Autodiff](https://docs.jax.dev/en/latest/advanced-autodiff.html#setup)

## Concepts In action:



In [ ]:
import jax
import jax.numpy as jnp

# Gradient Manipulations

## Gradient Stopping

### Example

Here we will cover gradient-stopping, which is useful when you want to skip backpropagating gradients through some subset of the computational graph. 

Note: this is akin to using `detach` in PyTorch to stop calculating the gradient. 

Let's pretend we have the following simple computation graph:

<img src="../assets/simple_graph.png" alt="drawing" width="400"/>

Say `r_2` should be some fixed relation - it was some "hard-crafted" feature specified by an external expert and we want to keep it. Unsurprisingly, the gradient will be different if we block it or allow the gradient to be calculated through it. Let's see that in action.

In [ ]:
import jax
import jax.numpy as jnp

def l_node(x, y):
    return (x - 0.5) * (y + 0.5)

def r_node(x, y):
    r_1 = x * y
    r_2 = x / y
    return r_1 + r_2

def r_node_blocked(x, y):
    r_1 = x * y
    r_2 = jax.lax.stop_gradient(x / y)
    return r_1 + r_2

def evaluation_fn(x, y, use_blocked_right):
    l_contrib = l_node(x, y)
    if use_blocked_right:
        r_contrib = r_node_blocked(x, y)
    else:
        r_contrib = r_node(x, y)
    return l_contrib + r_contrib

# Create some input values
x = 3.0
y = 4.0

grad_fn = jax.grad(evaluation_fn, argnums=(0, 1))
for should_block in [True, False]:
    dx, dy = grad_fn(x, y, should_block)
    print(f"{should_block=}\t{dx=}\t{dy=}")

### Real World Applications of Gradient Stopping

In principle, this is similar to:

- [Deep Deterministic Policy Gradients](https://spinningup.openai.com/en/latest/algorithms/ddpg.html#id1), where you have you pass your critic's prediction of the Q-value of the state to the actor, but do not want to update the actor's gradients with respect to the critic's prediction. 

- [Bootstrap Your Own Latents](https://arxiv.org/pdf/2006.07733)

<img src="../assets/byol.png" alt="drawing" width="800"/>

where you do self-supervised learning. The target is updated as an EMA of the online network.

## Skipping application of gradient

When you work with [Flax](https://flax.readthedocs.io/en/stable/) or [Equinox](https://github.com/patrick-kidger/equinox) you'll likely use the [optax](https://optax.readthedocs.io/en/latest/index.html) library to apply the gradient updates via algorithms such as adam, adagrad, etc. It's best to see this in action, on your own, so let's test it out here.

### Example

Note: for further reading or if you're interested in PyTorch check out [PyTorch Gradient Manipulation 1 - Ian Quah](https://ianq.ai/pytorch-gradients-pt1/) for some relatively general concepts that might apply to your own work.

# Comprehensive Concept Evaluation

## Scenario

You've got a model that can be represented by the following computational graph:

<img src="../assets/grad_comprehensive_graph.png" height="200" width="300"/>

where we're learning over the Conv. Layer and one of the dense layers - the other has been fixed by some constraint. Let's also implement a momentum optimizer. As such, there are 3 concepts at play here:

0) **gradient skipping** in the `MomentumOptimizer`
1) **gradient masking** in the graph definition (in the `predict`)
2) **value-and-grad** in the `train-loop`

## Data And Parameter Setup

No need to worry about this. Skip to the next section

In [ ]:
from jax import random
import jax.numpy as jnp
import jax

DENSE_SHAPE = (300, 10)
IMG_HEIGHT = IMG_WIDTH = 10
OUT_KERNELS = IN_KERNELS = 3
KERNEL_HEIGHT = KERNEL_WIDTH = 3

####################################
# Model Parameter Setup
####################################

def setup_params():
    # IOHW format for single-sample convolution - input, output, height, width
    kernel = jnp.zeros((IN_KERNELS, OUT_KERNELS, KERNEL_HEIGHT, KERNEL_WIDTH), dtype=jnp.float32)
    kernel += jnp.array([[1, 1, 0],
                         [1, 0,-1],
                         [0,-1,-1]])[:, :, jnp.newaxis, jnp.newaxis]

    key = random.PRNGKey(42)
    dense_frozen_key, dense_learnable_key, output_key = random.split(key, num=3)
    
    dense_frozen = random.normal(dense_frozen_key, shape=DENSE_SHAPE)
    dense_learnable = random.normal(dense_learnable_key, shape=DENSE_SHAPE)
    dense_output = random.normal(output_key, shape=(DENSE_SHAPE[-1], 1))
    return {
        "conv_kernel": kernel,
        "fixed_dense": dense_frozen,
        "learnable_dense": dense_learnable,
        "final": dense_output
    }

INITIAL_PARAMS = setup_params()

####################################
# Train Data Setup
####################################
N_IMAGES = 2
# NHWC layout - Batch Size (N), Height, Width, #Channels
X = jnp.zeros((N_IMAGES, IMG_HEIGHT, IMG_WIDTH, 3), dtype=jnp.float32)

# First Image
X = X.at[0, 0:2, 0:2, 0].set(1.0)
X = X.at[0, 3:5, 3:5, 1].set(1.0)
X = X.at[0, 7:9, 7:9, 2].set(1.0)

X = X.at[1, 0:2, 0:2, 2].set(1.0)
X = X.at[1, 3:5, 3:5, 1].set(1.0)
X = X.at[1, 7:9, 7:9, 0].set(1.0)

# Classify if the top-left contains a red square
y = jnp.zeros((N_IMAGES))
y = y.at[0].set(1.0)  

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 3)
axs[0].imshow(X[0])
axs[1].imshow(X[1])
axs[2].imshow(INITIAL_PARAMS["conv_kernel"][:, :, 0, 0])

## Momentum Optimizer

We do a simple one-step optimizer

In [ ]:
from dataclasses import dataclass

@dataclass
class MomentumOptimizer:
    momentum_factor: float
    
    def __post_init__(self):
        self.momentum = {}

    def update(
        self, 
        params: dict[str, jnp.ndarray],
        grads: dict[str, jnp.ndarray],
        learning_rate: float, 
        ignore: dict[str, bool] | None = None
    ) -> dict[str, jnp.ndarray]:
        """  
        Accepts a params dict that has the same "shape" as grads i.e. 
        - has all the same keys
        - has gradients that are the same shape as the parameter tensors

        Returns the updated params
        """
        if ignore is not None:
            assert set(ignore.keys()) == set(params.keys()) == set(grads.keys())
        else:
            assert set(params.keys()) == set(grads.keys())

        new_params = {}
        
        for k, v in params.items():
            should_ignore = ignore is not None and ignore.get(k, False)
            
            if not should_ignore:
                if k not in self.momentum:
                    self.momentum[k] = jnp.zeros_like(grads[k])
                
                # Update momentum: momentum = momentum_factor * prev_momentum + current_grad
                self.momentum[k] = self.momentum_factor * self.momentum[k] + grads[k]
                
                # Update parameters: param = param - learning_rate * momentum
                new_params[k] = v + learning_rate * self.momentum[k]
            else:
                # Don't update ignored parameters
                new_params[k] = v
    
        return new_params

## Gradient Masking

Here is an incorrect implementation. Fix it! 

In [ ]:
from jax import lax

def _predict_single_logits(x: jnp.ndarray, params: dict[str, jnp.ndarray]) -> jnp.ndarray:
    """
    Predict for a single sample (HWC format)
    P.s. if you're reading this, you might find it cool how you can 
        represent convolutions as einsums: https://openreview.net/forum?id=cDS8WxnMVP
    """
    # Convert HWC to NCHW for convolution (add batch dimension), but dim is just 1. 
    x_nchw = jnp.transpose(x, [2, 0, 1])[None, ...]

    # Convolution: NCHW * OIHW -> NOHW
    out = lax.conv(x_nchw,
                   jnp.transpose(params["conv_kernel"], [1, 0, 2, 3]), # rhs = IOHW -> OIHW
                   (1, 1),
                   'SAME')
    out = out.ravel()
    out_frozen = jax.lax.stop_gradient(out @ params["fixed_dense"])
    out_learnable = out @ params["learnable_dense"]
    return (out_frozen + out_learnable) @ params["final"]

predict_logits = jax.vmap(_predict_single_logits, in_axes=(0, None))

## Train-Loop

instantiate the optimizer and make sure you set up the mask properly! 

In [ ]:
@jax.jit
def _log_likelihood_from_logits(y, X, params):
    """
    Relies on log_sigmoid, which is basically the log-sum-exp trick
    https://gregorygundersen.com/blog/2020/02/09/log-sum-exp/
    """
    logits = predict_logits(X, params)
    return jnp.sum(y * jax.nn.log_sigmoid(logits) + 
                   (1 - y) * jax.nn.log_sigmoid(-logits))

    
log_likelihood_and_grad = jax.value_and_grad(_log_likelihood_from_logits, argnums=2)

def train_loop(params, optimizer, lr, X, y, ignore_mask):
    for i in range(100):
        loss, grads = log_likelihood_and_grad(y, X, params)
        params = optimizer.update(params, grads, lr, ignore_mask)
        if i % 10 == 0 and i > 1:
            print(f"At iteration: {i}, loss was: {loss}")
    loss, _ = log_likelihood_and_grad(y, X, params)
    print(f"At iteration: {i}, loss was: {loss}")
    return params

MOMENTUM_FACTOR = 0.01
LR = 0.01
optimizer = MomentumOptimizer(MOMENTUM_FACTOR)

ignore_mask = {
    "conv_kernel": False,
    "fixed_dense": True,
    "learnable_dense": False,
    "final": False
}

final_params = train_loop(INITIAL_PARAMS, optimizer, LR, X, y, ignore_mask)

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(3, 3)
fig.suptitle("Visualize the nearly learned kern")
axs[0, 0].imshow(final_params["conv_kernel"][:, :, 0, 0])
axs[0, 1].imshow(final_params["conv_kernel"][:, :, 0, 1])
axs[0, 2].imshow(final_params["conv_kernel"][:, :, 0, 2])

axs[1, 0].imshow(final_params["conv_kernel"][:, :, 1, 0])
axs[1, 1].imshow(final_params["conv_kernel"][:, :, 1, 1])
axs[1, 2].imshow(final_params["conv_kernel"][:, :, 1, 2])

axs[2, 0].imshow(final_params["conv_kernel"][:, :, 2, 0])
axs[2, 1].imshow(final_params["conv_kernel"][:, :, 2, 1])
axs[2, 2].imshow(final_params["conv_kernel"][:, :, 2, 2])

# Closing

That's all we've got for the gradient work! For more reading check out 

- [Custom Derivative Rules](https://docs.jax.dev/en/latest/notebooks/Custom_derivative_rules_for_Python_code.html) 
- [Jax autodiff_cookbook](https://docs.jax.dev/en/latest/notebooks/autodiff_cookbook.html#gradients) - details in-depth how to go from the high-level `grad` function to the in-the-weeds `jvp` and `vjp`
- [From JVP to VJP (Part I): In Pictures](https://pasteurlabs.ai/insights/jax)
- [From JVP to VJP (Part II)](https://pasteurlabs.ai/insights/jax-2)